In [1]:
import pandas as pd

In [2]:
# https://www.kaggle.com/datasets/datascikhan/e-commerce-customer-churn-2026

# Link for the Dataset 

In [3]:
df=pd.read_csv("E-commerce_Customer_Churn_2026.csv")

In [4]:
df.head(5)

,churn_id,customer_id,customer_segment,customer_country,customer_city,is_churned,churn_flag,churn_date,days_since_churn,churn_reason,...,customer_retention_cost_usd,recovery_revenue_usd,dataset_year,churn_rate_indicator,customer_value_category,churn_risk_category,churn_revenue_impact_usd,customer_health_score,customer_health_category,retention_probability
0,CHURN-000001,CUST-020712,Consumer,Australia,Brisbane,Yes,1,2026-05-18 00:00:00,23,Found Alternative,...,29.33,346.60,2026,Medium,High Value,Medium,10351.348,75,Good,0.10
1,CHURN-000002,CUST-044206,Small Business,Saudi Arabia,Khobar,No,0,Not Applicable,0,Not Applicable,...,49.22,403.14,2026,Low,High Value,Low,0.000,100,Excellent,0.66
2,CHURN-000003,CUST-024093,Small Business,Mexico,Leon,No,0,Not Applicable,0,Not Applicable,...,198.06,4.20,2026,Low,High Value,Low,0.000,100,Excellent,0.76
3,CHURN-000004,CUST-037777,Enterprise,Saudi Arabia,Jeddah,No,0,Not Applicable,0,Not Applicable,...,190.31,222.46,2026,Low,High Value,Very Low,0.000,100,Excellent,0.98
4,CHURN-000005,CUST-016248,Premium,Pakistan,Multan,Yes,1,2026-05-28 00:00:00,7,Poor Customer Service,...,26.49,489.02,2026,Medium,High Value,Medium,14212.088,80,Good,0.10


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 48 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   churn_id                     50000 non-null  str    
 1   customer_id                  50000 non-null  str    
 2   customer_segment             50000 non-null  str    
 3   customer_country             50000 non-null  str    
 4   customer_city                50000 non-null  str    
 5   is_churned                   50000 non-null  str    
 6   churn_flag                   50000 non-null  int64  
 7   churn_date                   50000 non-null  str    
 8   days_since_churn             50000 non-null  int64  
 9   churn_reason                 50000 non-null  str    
 10  churn_category               50000 non-null  str    
 11  churn_risk_score             50000 non-null  int64  
 12  churn_probability            50000 non-null  float64
 13  subscription_type          

In [6]:
# to count total_number of missing values per column 
df.isnull().sum()

churn_id                           0
customer_id                        0
customer_segment                   0
customer_country                   0
customer_city                      0
is_churned                         0
churn_flag                         0
churn_date                         0
days_since_churn                   0
churn_reason                       0
churn_category                     0
churn_risk_score                   0
churn_probability                  0
subscription_type              13142
customer_age_months                0
total_spend_usd                    0
avg_order_value_usd                0
num_purchases                      0
satisfaction_score                 0
satisfaction_level                 0
support_tickets                    0
returns_count                      0
days_since_last_purchase           0
customer_lifetime_value_usd        0
engagement_level                   0
prevention_method              36874
support_interactions               0
l

In [7]:
# count number of completely duplicated row
df.duplicated().sum()

np.int64(0)

In [8]:
# 1. Convert 'churn_date' to datetime format
df['churn_date'] = pd.to_datetime(df['churn_date'], errors='coerce')

# 2. Convert numerical columns (strip any currency symbols if present, though these look clean)
numeric_cols = [
    'customer_retention_cost_usd', 
    'recovery_revenue_usd', 
    'churn_revenue_impact_usd', 
    'customer_health_score', 
    'retention_probability'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. Standardize categorical text columns (lowercase and strip whitespace)
text_cols = ['customer_segment', 'customer_country', 'customer_city', 'churn_reason', 'customer_health_category']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().str.strip()

# 4. Handle Missing Values
# For columns like 'churn_reason' or 'days_since_churn', 'Not Applicable' or NaNs might be present for non-churned customers.
# You can fill missing numeric values or categorical ones appropriately:
df['churn_reason'] = df['churn_reason'].fillna('not applicable')

In [9]:
# 1. Convert 'churn_date' to datetime format
df['churn_date'] = pd.to_datetime(df['churn_date'], errors='coerce')

# 2. Convert numerical columns (strip any currency symbols if present, though these look clean)
numeric_cols = [
    'customer_retention_cost_usd', 
    'recovery_revenue_usd', 
    'churn_revenue_impact_usd', 
    'customer_health_score', 
    'retention_probability'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. Standardize categorical text columns (lowercase and strip whitespace)
text_cols = ['customer_segment', 'customer_country', 'customer_city', 'churn_reason', 'customer_health_category']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().str.strip()

# 4. Handle Missing Values
# For columns like 'churn_reason' or 'days_since_churn', 'Not Applicable' or NaNs might be present for non-churned customers.
# You can fill missing numeric values or categorical ones appropriately:
df['churn_reason'] = df['churn_reason'].fillna('not applicable')

In [10]:
# 1. Create a binary indicator for churn if not already properly formatted
# (Assuming 'is_churned' or 'churn_flag' holds Yes/No or 1/0)
df['is_churned_numeric'] = df['is_churned'].apply(lambda x: 1 if str(x).lower() in ['yes', '1', 'true'] else 0)

# 2. Extract date features from 'churn_date' for seasonal/trend analysis
df['churn_year'] = df['churn_date'].dt.year
df['churn_month'] = df['churn_date'].dt.month

# 3. Calculate Net Revenue Impact of Churn (Revenue Impact minus Recovery Revenue)
df['net_churn_loss'] = df['churn_revenue_impact_usd'] - df['recovery_revenue_usd']

In [11]:
# Check summary statistics for numerical columns
print(df.describe())

# Check class distribution of your target variable (Churn vs Non-Churn)
print(df['is_churned'].value_counts(normalize=True))

         churn_flag                  churn_date  days_since_churn  \
count  50000.000000                       20653      50000.000000   
mean       0.413060  2026-07-02 05:14:52.286834         37.193620   
min        0.000000         2026-01-01 00:00:00          0.000000   
25%        0.000000         2026-04-03 00:00:00          0.000000   
50%        0.000000         2026-07-03 00:00:00          0.000000   
75%        1.000000         2026-10-02 00:00:00         72.000000   
max        1.000000         2026-12-31 00:00:00        179.000000   
std        0.492388                         NaN         55.352953   

       churn_risk_score  churn_probability  customer_age_months  \
count      50000.000000       50000.000000         50000.000000   
mean          41.159080           0.411591            30.134580   
min            0.000000           0.000000             1.000000   
25%           22.000000           0.220000            16.000000   
50%           40.000000           0.400000 

In [12]:
# Check total missing values per column
print(df.isnull().sum())

churn_id                           0
customer_id                        0
customer_segment                   0
customer_country                   0
customer_city                      0
is_churned                         0
churn_flag                         0
churn_date                     29347
days_since_churn                   0
churn_reason                       0
churn_category                     0
churn_risk_score                   0
churn_probability                  0
subscription_type              13142
customer_age_months                0
total_spend_usd                    0
avg_order_value_usd                0
num_purchases                      0
satisfaction_score                 0
satisfaction_level                 0
support_tickets                    0
returns_count                      0
days_since_last_purchase           0
customer_lifetime_value_usd        0
engagement_level                   0
prevention_method              36874
support_interactions               0
l

In [13]:
# Fill structural or categorical gaps safely
df['subscription_type'] = df['subscription_type'].fillna('Unknown')
df['prevention_method'] = df['prevention_method'].fillna('None')
df['social_media_engagement'] = df['social_media_engagement'].fillna(0)

In [14]:
# Quick statistical check for anomalies
print(df[['customer_retention_cost_usd', 'churn_revenue_impact_usd', 'customer_health_score']].describe())

       customer_retention_cost_usd  churn_revenue_impact_usd  \
count                 50000.000000              50000.000000   
mean                     99.903609               4135.546160   
std                      57.774781               6158.312439   
min                       0.000000                  0.000000   
25%                      49.767500                  0.000000   
50%                     100.105000                  0.000000   
75%                     149.802500               7923.600000   
max                     200.000000              19999.320000   

       customer_health_score  
count           50000.000000  
mean               88.604100  
std                17.484367  
min                25.000000  
25%                80.000000  
50%               100.000000  
75%               100.000000  
max               100.000000  


In [15]:
# Check unique values in key segments
print(df['customer_segment'].value_counts())
print(df['customer_health_category'].value_counts())

customer_segment
consumer          22320
premium           12565
small business     9983
enterprise         5132
Name: count, dtype: int64
customer_health_category
excellent    36874
good          8347
fair          3557
poor          1222
Name: count, dtype: int64


In [16]:
import pandas as pd
import numpy as np

def check_data_readiness(df):
    print("=" * 40)
    print("      DATA READINESS AUDIT REPORT       ")
    print("=" * 40)
    
    issues_found = False
    
    # 1. Dataset Dimensions
    print(f"📁 Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns\n")
    
    # 2. Check for Duplicate Rows
    duplicates = df.duplicated().sum()
    if duplicates > 0:
        print(f"⚠️️ Warning: {duplicates:,} duplicate rows found.")
        issues_found = True
    else:
        print("✅ No duplicate rows detected.")
        
    # 3. Check for Missing Values
    missing_counts = df.isnull().sum()
    total_missing = missing_counts.sum()
    if total_missing > 0:
        print(f"ℹ️ Total Missing Values across DataFrame: {total_missing:,}")
        print("   Breakdown of columns with missing data:")
        print(missing_counts[missing_counts > 0].to_string())
        print("   (Note: Ensure these are expected structural NaNs, like un-churned users missing churn dates).")
    else:
        print("✅ Zero missing values across all columns.")
        
    # 4. Check for Infinite Numerical Values
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    inf_counts = np.isinf(df[numeric_cols]).sum().sum()
    if inf_counts > 0:
        print(f"⚠️ Warning: {inf_counts:,} infinite (inf/-inf) values found in numeric columns.")
        issues_found = True
    else:
        print("✅ No infinite values found in numeric features.")
        
    # 5. Check for Constant Columns (Zero Variance)
    constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
    if constant_cols:
        print(f"⚠️ Warning: Constant columns with zero variance found: {constant_cols}")
        issues_found = True
    else:
        print("✅ All columns have healthy variance (more than 1 unique value).")
        
    # 6. Final Conclusion
    print("\n" + "=" * 40)
    if issues_found:
        print("❌ STATUS: Review the warnings above. Minor cleaning may still be required.")
    else:
        print("✅ STATUS: Data is structurally sound and ready for Exploratory Data Analysis (EDA)!")
    print("=" * 40)

# Run the audit on your current dataframe
check_data_readiness(df)

      DATA READINESS AUDIT REPORT       
📁 Dataset Shape: 50,000 rows x 52 columns

✅ No duplicate rows detected.
ℹ️ Total Missing Values across DataFrame: 88,041
   Breakdown of columns with missing data:
churn_date     29347
churn_year     29347
churn_month    29347
   (Note: Ensure these are expected structural NaNs, like un-churned users missing churn dates).
✅ No infinite values found in numeric features.
⚠️ Warning: Constant columns with zero variance found: ['dataset_year', 'churn_year']

❌ STATUS: Review the warnings above. Minor cleaning may still be required.


In [17]:
# 1. Drop zero-variance columns as they provide no analytical or predictive value
cols_to_drop = ['dataset_year', 'churn_year']
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# 2. Fill structural NaNs for month and date with safe placeholders for filtering/analysis
df['churn_month'] = df['churn_month'].fillna(0)  # 0 or 'None' for non-churned users
df['churn_date'] = pd.to_datetime(df['churn_date'], errors='coerce') # Keeps NaT for active users

print("Cleaned DataFrame Shape:", df.shape)
print("✅ Data is now 100% ready for Exploratory Data Analysis!")

Cleaned DataFrame Shape: (50000, 50)
✅ Data is now 100% ready for Exploratory Data Analysis!


In [18]:
df.to_csv('cleaned_customer_churn_data.csv', index=False)

print("✅ Data successfully saved to CSV!")

✅ Data successfully saved to CSV!
